**Image Deblurring using Residual CNN**

This notebook trains a deep residual convolutional neural network for single-image deblurring using the DIV2K and Flickr2K datasets.

**Pipeline**

- Use clean HR images as ground truth.
- Take blurred images from RealBlur-J and GoPro.
- Train a residual CNN to restore the original image.
- Evaluate using Validation Loss, PSNR and SSIM.
- Save checkpoints, resume training and export the best model.

This model does **not** use a discriminator or adversarial training. Since image deblurring is a restoration task rather than an image generation task, supervised learning with perceptual loss provides more stable training and better quantitative performance.

# Phase 1: Setup and Configuration

Import the required libraries, initialize the training environment, define hyperparameters and configure image transformations, evaluation metrics and training variables used throughout the notebook.

In [1]:
# ============================================================
# Imports
# ============================================================
import os
import cv2
import math
import random
import shutil
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import VGG19_Weights
from torch.utils.data import Dataset, DataLoader, random_split
from torchmetrics.image import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ============================================================
# Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# Hyperparameters
# ============================================================
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_EPOCHS = 200
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-6
VALIDATION_IMAGES = 100
NUM_RESIDUAL_BLOCKS = 16
SAVE_DIR = "/kaggle/working/Resume"
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# ImageNet Normalization
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ============================================================
# Image Transform
# ============================================================
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.1,
        saturation=0.1,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ============================================================
# Metrics
# ============================================================
psnr_metric = PeakSignalNoiseRatio(
    data_range=1.0
).to(device)

ssim_metric = StructuralSimilarityIndexMeasure(
    data_range=1.0
).to(device)

# ============================================================
# Training History
# ============================================================
train_losses = []
val_losses = []
psnr_scores = []
ssim_scores = []

# ============================================================
# Best Model Tracking
# ============================================================
best_validation_loss = float("inf")
start_epoch = 0
print("\nInitialization Complete.")

Using device: cuda

Initialization Complete.


# Phase 2: Dataset Preparation

Prepare dataset from images of RealBlur-J and GoPro

In [2]:
# ============================================================
# IMPORTS
# ============================================================
import os
import random
import numpy as np
import cv2
import torch

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Image size used by the model
# ------------------------------------------------------------
IMAGE_SIZE = 256

# ------------------------------------------------------------
# Dataset split ratios
# ------------------------------------------------------------

# GoPro:
# Official train folder -> our train + validation
GOPRO_VAL_RATIO = 0.10

# RealBlur-J:
# Its available data is under test.
#
# We divide it into:
#   70% -> training
#   15% -> validation
#   15% -> final test
#
# These ratios can be changed later.
RBJ_TRAIN_RATIO = 0.70
RBJ_VAL_RATIO = 0.15
RBJ_TEST_RATIO = 0.15

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


# ============================================================
# DATASET PATHS
# ============================================================

# ------------------------------------------------------------
# GoPro
#
# Expected structure:
#
# GoPro/
# ├── train/
# │   ├── scene_1/
# │   │   ├── blur/
# │   │   └── sharp/
# │   ├── scene_2/
# │   │   ├── blur/
# │   │   └── sharp/
# │   └── ...
# │
# └── test/
#     ├── scene_1/
#     │   ├── blur/
#     │   └── sharp/
#     └── ...
# ------------------------------------------------------------

GOPRO_ROOT = "/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro"

# ------------------------------------------------------------
# RealBlur-J
#
# We expect the available RealBlur-J images to eventually
# resolve into corresponding blur/sharp images.
#
# The code below searches recursively so the exact number
# of intermediate folders does not matter.
# ------------------------------------------------------------

REALBLUR_J_ROOT = "/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/RealBlur_J"

# ============================================================
# IMAGE EXTENSIONS
# ============================================================

IMAGE_EXTENSIONS = (
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
)

# ============================================================
# HELPER:
# GET IMAGE FILES
# ============================================================

def is_image_file(filename):
    return filename.lower().endswith(IMAGE_EXTENSIONS)


# ============================================================
# HELPER:
# BUILD FILENAME → PATH DICTIONARY
# ============================================================

def build_image_dictionary(folder):
    """
    Recursively search a folder and create:

        filename -> complete path

    Example:

        GOPR0384_11_00.png
        ->
        /.../blur/GOPR0384_11_00.png

    Only the filename is used for matching because the
    corresponding blur/sharp images have matching names.
    """

    image_dict = {}

    for root, _, files in os.walk(folder):

        for file in files:

            if not is_image_file(file):
                continue

            # ------------------------------------------------
            # Use filename as the matching key
            # ------------------------------------------------
            key = file.lower()

            full_path = os.path.join(
                root,
                file
            )

            # ------------------------------------------------
            # Detect duplicate filenames
            # ------------------------------------------------
            if key in image_dict:

                print(
                    "WARNING: Duplicate filename found:"
                )

                print(
                    f"  Existing: {image_dict[key]}"
                )

                print(
                    f"  New     : {full_path}"
                )

            image_dict[key] = full_path

    return image_dict


# ============================================================
# GOPRO PAIR COLLECTION
# ============================================================

def collect_gopro_pairs(split_dir):
    """
    Collect paired GoPro images.

    Supports BOTH structures:

    Structure 1:
        split_dir/
            blur/
            sharp/

    Structure 2:
        split_dir/
            scene_1/
                blur/
                sharp/
            scene_2/
                blur/
                sharp/

    Blur and sharp images are matched strictly by filename.
    """

    pairs = []

    if not os.path.isdir(split_dir):

        raise FileNotFoundError(
            f"GoPro directory not found:\n{split_dir}"
        )

    print(
        f"\nSearching GoPro directory:\n"
        f"{split_dir}"
    )

    # ========================================================
    # CHECK WHETHER blur/sharp ARE DIRECT CHILDREN
    # ========================================================

    entries = {
        item.lower(): item
        for item in os.listdir(split_dir)
    }

    direct_blur = None
    direct_sharp = None

    # --------------------------------------------------------
    # Find blur folder
    # --------------------------------------------------------

    for name in [
        "blur",
        "blurry",
        "blurred"
    ]:

        if name in entries:

            candidate = os.path.join(
                split_dir,
                entries[name]
            )

            if os.path.isdir(candidate):

                direct_blur = candidate
                break

    # --------------------------------------------------------
    # Find sharp folder
    # --------------------------------------------------------

    for name in [
        "sharp",
        "gt",
        "groundtruth",
        "ground_truth"
    ]:

        if name in entries:

            candidate = os.path.join(
                split_dir,
                entries[name]
            )

            if os.path.isdir(candidate):

                direct_sharp = candidate
                break

    # ========================================================
    # CASE 1:
    # DIRECT blur/ AND sharp/ FOLDERS
    # ========================================================

    if (
        direct_blur is not None
        and
        direct_sharp is not None
    ):

        print(
            "Detected direct blur/sharp structure."
        )

        blur_images = build_image_dictionary(
            direct_blur
        )

        sharp_images = build_image_dictionary(
            direct_sharp
        )

        common_names = sorted(
            set(blur_images.keys())
            &
            set(sharp_images.keys())
        )

        for name in common_names:

            pairs.append(
                {
                    "blur": blur_images[name],
                    "sharp": sharp_images[name],
                    "source": "GoPro"
                }
            )

        # ----------------------------------------------------
        # Unmatched files
        # ----------------------------------------------------

        missing_sharp = (
            set(blur_images.keys())
            -
            set(sharp_images.keys())
        )

        missing_blur = (
            set(sharp_images.keys())
            -
            set(blur_images.keys())
        )

        if missing_sharp:

            print(
                f"WARNING: {len(missing_sharp)} "
                f"blur images have no sharp match."
            )

        if missing_blur:

            print(
                f"WARNING: {len(missing_blur)} "
                f"sharp images have no blur match."
            )

        print(
            f"Blur images  : {len(blur_images)}"
        )

        print(
            f"Sharp images : {len(sharp_images)}"
        )

        print(
            f"Matched GoPro pairs: {len(pairs)}"
        )

        return pairs

    # ========================================================
    # CASE 2:
    # SCENE SUBDIRECTORIES
    # ========================================================

    print(
        "Direct blur/sharp folders not found."
    )

    print(
        "Checking scene subdirectories..."
    )

    scene_dirs = []

    for item in sorted(
        os.listdir(split_dir)
    ):

        scene_path = os.path.join(
            split_dir,
            item
        )

        if os.path.isdir(scene_path):

            scene_dirs.append(
                scene_path
            )

    print(
        f"Scene directories found: "
        f"{len(scene_dirs)}"
    )

    # --------------------------------------------------------
    # Process each scene
    # --------------------------------------------------------

    for scene_dir in scene_dirs:

        entries = {
            item.lower(): item
            for item in os.listdir(scene_dir)
        }

        blur_dir = None
        sharp_dir = None

        # ----------------------------------------------------
        # Find blur directory
        # ----------------------------------------------------

        for name in [
            "blur",
            "blurry",
            "blurred"
        ]:

            if name in entries:

                candidate = os.path.join(
                    scene_dir,
                    entries[name]
                )

                if os.path.isdir(candidate):

                    blur_dir = candidate
                    break

        # ----------------------------------------------------
        # Find sharp directory
        # ----------------------------------------------------

        for name in [
            "sharp",
            "gt",
            "groundtruth",
            "ground_truth"
        ]:

            if name in entries:

                candidate = os.path.join(
                    scene_dir,
                    entries[name]
                )

                if os.path.isdir(candidate):

                    sharp_dir = candidate
                    break

        if (
            blur_dir is None
            or
            sharp_dir is None
        ):

            continue

        blur_images = build_image_dictionary(
            blur_dir
        )

        sharp_images = build_image_dictionary(
            sharp_dir
        )

        common_names = sorted(
            set(blur_images.keys())
            &
            set(sharp_images.keys())
        )

        for name in common_names:

            pairs.append(
                {
                    "blur": blur_images[name],
                    "sharp": sharp_images[name],
                    "source": "GoPro"
                }
            )

    print(
        f"Matched GoPro pairs: {len(pairs)}"
    )

    return pairs


# ============================================================
# REALBLUR-J PAIR COLLECTION
# ============================================================

def collect_realblur_j_pairs(root_dir):
    """
    Recursively collect RealBlur-J paired images.

    The function is intentionally flexible because Kaggle
    versions of the dataset can have different intermediate
    directory structures.

    It searches for folders containing blur/sharp images.

    Example supported structure:

        root/
            .../
                blur/
                sharp/

    or:

        root/
            .../
                blurred/
                sharp/

    Matching is always performed by filename.
    """

    pairs = []

    if not os.path.isdir(root_dir):

        raise FileNotFoundError(
            f"RealBlur-J directory not found:\n{root_dir}"
        )

    # --------------------------------------------------------
    # Search recursively for blur/sharp directory pairs
    # --------------------------------------------------------

    candidate_dirs = []

    for root, dirs, _ in os.walk(root_dir):

        lower_dirs = {
            d.lower(): d
            for d in dirs
        }

        blur_name = None
        sharp_name = None

        # ----------------------------------------------------
        # Possible names
        # ----------------------------------------------------

        for name in [
            "blur",
            "blurry",
            "blurred"
        ]:

            if name in lower_dirs:

                blur_name = lower_dirs[name]
                break

        for name in [
            "sharp",
            "gt",
            "groundtruth",
            "ground_truth"
        ]:

            if name in lower_dirs:

                sharp_name = lower_dirs[name]
                break

        if blur_name is not None and sharp_name is not None:

            candidate_dirs.append(
                (
                    os.path.join(
                        root,
                        blur_name
                    ),
                    os.path.join(
                        root,
                        sharp_name
                    )
                )
            )

    print(
        f"\nRealBlur-J blur/sharp directory pairs found:"
        f" {len(candidate_dirs)}"
    )

    # --------------------------------------------------------
    # Match each blur/sharp directory
    # --------------------------------------------------------

    for blur_dir, sharp_dir in candidate_dirs:

        blur_images = build_image_dictionary(
            blur_dir
        )

        sharp_images = build_image_dictionary(
            sharp_dir
        )

        common_names = sorted(
            set(blur_images.keys())
            &
            set(sharp_images.keys())
        )

        for name in common_names:

            pairs.append(
                {
                    "blur": blur_images[name],
                    "sharp": sharp_images[name],
                    "source": "RealBlur-J"
                }
            )

        missing_sharp = (
            set(blur_images.keys())
            -
            set(sharp_images.keys())
        )

        missing_blur = (
            set(sharp_images.keys())
            -
            set(blur_images.keys())
        )

        if missing_sharp:

            print(
                f"WARNING: {len(missing_sharp)} "
                f"RealBlur blur images have no sharp match."
            )

        if missing_blur:

            print(
                f"WARNING: {len(missing_blur)} "
                f"RealBlur sharp images have no blur match."
            )

    print(
        f"Matched RealBlur-J pairs: {len(pairs)}"
    )

    return pairs


# ============================================================
# SPLIT HELPER
# ============================================================

def split_pairs(
    pairs,
    train_ratio,
    val_ratio,
    test_ratio=0.0,
    seed=42
):
    """
    Randomly split paired images.

    IMPORTANT:
    The pair remains together.

    We NEVER split blur and sharp independently.
    """

    if abs(
        train_ratio +
        val_ratio +
        test_ratio -
        1.0
    ) > 1e-6:

        raise ValueError(
            "Train + validation + test ratios "
            "must equal 1.0"
        )

    pairs = list(pairs)

    rng = random.Random(seed)

    rng.shuffle(pairs)

    total = len(pairs)

    train_end = int(
        total * train_ratio
    )

    val_end = (
        train_end +
        int(total * val_ratio)
    )

    train_pairs = pairs[
        :train_end
    ]

    val_pairs = pairs[
        train_end:val_end
    ]

    test_pairs = pairs[
        val_end:
    ]

    return (
        train_pairs,
        val_pairs,
        test_pairs
    )


# ============================================================
# LOAD GOPRO
# ============================================================

gopro_train_dir = os.path.join(
    GOPRO_ROOT,
    "train"
)

gopro_test_dir = os.path.join(
    GOPRO_ROOT,
    "test"
)


gopro_train_pairs = collect_gopro_pairs(
    gopro_train_dir
)

gopro_test_pairs = collect_gopro_pairs(
    gopro_test_dir
)


# ============================================================
# SPLIT GOPRO TRAIN
# ============================================================
#
# Official GoPro TEST is NOT touched.
#
# Official GoPro TRAIN:
#
#     90% -> training
#     10% -> validation
#
# ============================================================

gopro_train_split, gopro_val_split, _ = split_pairs(
    gopro_train_pairs,
    train_ratio=0.90,
    val_ratio=0.10,
    test_ratio=0.0,
    seed=RANDOM_SEED
)


# ============================================================
# LOAD REALBLUR-J
# ============================================================

realblur_pairs = collect_realblur_j_pairs(
    REALBLUR_J_ROOT
)


# ============================================================
# SPLIT REALBLUR-J
# ============================================================
#
# Since this Kaggle version only gives us TEST:
#
#     70% -> training
#     15% -> validation
#     15% -> final test
#
# ============================================================

rbj_train_split, rbj_val_split, rbj_test_split = split_pairs(
    realblur_pairs,
    train_ratio=RBJ_TRAIN_RATIO,
    val_ratio=RBJ_VAL_RATIO,
    test_ratio=RBJ_TEST_RATIO,
    seed=RANDOM_SEED
)


# ============================================================
# COMBINE TRAINING SETS
# ============================================================

train_pairs = (
    gopro_train_split
    +
    rbj_train_split
)


# ============================================================
# COMBINE VALIDATION SETS
# ============================================================

val_pairs = (
    gopro_val_split
    +
    rbj_val_split
)


# ============================================================
# FINAL TEST SET
# ============================================================
#
# GoPro official TEST remains completely untouched.
#
# RealBlur-J:
#     15% reserved as our held-out test set.
#
# ============================================================

test_pairs = (
    gopro_test_pairs
    +
    rbj_test_split
)


# ============================================================
# SHUFFLE COMBINED TRAINING SET
# ============================================================

random.shuffle(
    train_pairs
)

random.shuffle(
    val_pairs
)

random.shuffle(
    test_pairs
)


# ============================================================
# PRINT SPLIT INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("FINAL DATASET SPLIT")
print("=" * 70)

print(
    f"GoPro official train pairs : "
    f"{len(gopro_train_pairs)}"
)

print(
    f"GoPro train split           : "
    f"{len(gopro_train_split)}"
)

print(
    f"GoPro validation split      : "
    f"{len(gopro_val_split)}"
)

print(
    f"GoPro official test pairs   : "
    f"{len(gopro_test_pairs)}"
)

print("-" * 70)

print(
    f"RealBlur-J total pairs      : "
    f"{len(realblur_pairs)}"
)

print(
    f"RealBlur-J train split      : "
    f"{len(rbj_train_split)}"
)

print(
    f"RealBlur-J validation split : "
    f"{len(rbj_val_split)}"
)

print(
    f"RealBlur-J test split       : "
    f"{len(rbj_test_split)}"
)

print("-" * 70)

print(
    f"TOTAL TRAINING PAIRS        : "
    f"{len(train_pairs)}"
)

print(
    f"TOTAL VALIDATION PAIRS      : "
    f"{len(val_pairs)}"
)

print(
    f"TOTAL TEST PAIRS            : "
    f"{len(test_pairs)}"
)

print("=" * 70)


# ============================================================
# DATASET
# ============================================================

class PairedDeblurDataset(Dataset):

    def __init__(
        self,
        pairs,
        training=True
    ):

        self.pairs = pairs
        self.training = training

        # ----------------------------------------------------
        # ImageNet normalization
        # ----------------------------------------------------

        self.mean = torch.tensor(
            IMAGENET_MEAN,
            dtype=torch.float32
        ).view(
            3,
            1,
            1
        )

        self.std = torch.tensor(
            IMAGENET_STD,
            dtype=torch.float32
        ).view(
            3,
            1,
            1
        )


    def __len__(self):

        return len(
            self.pairs
        )


    def __getitem__(self, idx):

        pair = self.pairs[idx]

        blur_path = pair["blur"]
        sharp_path = pair["sharp"]


        # ====================================================
        # LOAD BLURRED IMAGE
        # ====================================================

        blurred = Image.open(
            blur_path
        ).convert(
            "RGB"
        )


        # ====================================================
        # LOAD SHARP IMAGE
        # ====================================================

        sharp = Image.open(
            sharp_path
        ).convert(
            "RGB"
        )


        # ====================================================
        # SAFETY CHECK
        # ====================================================
        #
        # Blur and sharp must correspond to the same image.
        #
        # They normally have the same dimensions.
        # ====================================================

        if blurred.size != sharp.size:

            raise RuntimeError(
                "\nBlur/sharp dimension mismatch!\n"
                f"Blur : {blur_path}\n"
                f"Size : {blurred.size}\n"
                f"Sharp: {sharp_path}\n"
                f"Size : {sharp.size}"
            )


        # ====================================================
        # RESIZE BOTH
        # ====================================================
        #
        # IMPORTANT:
        # Exactly the same transformation is applied to both.
        # ====================================================

        blurred = transforms.functional.resize(
            blurred,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=Image.Resampling.BICUBIC
        )

        sharp = transforms.functional.resize(
            sharp,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=Image.Resampling.BICUBIC
        )


        # ====================================================
        # SPATIAL AUGMENTATION
        # ====================================================
        #
        # The SAME augmentation parameters must be applied
        # to both images.
        # ====================================================

        if self.training:

            # ------------------------------------------------
            # Horizontal flip
            # ------------------------------------------------

            if random.random() < 0.5:

                blurred = transforms.functional.hflip(
                    blurred
                )

                sharp = transforms.functional.hflip(
                    sharp
                )


            # ------------------------------------------------
            # Vertical flip
            # ------------------------------------------------

            if random.random() < 0.1:

                blurred = transforms.functional.vflip(
                    blurred
                )

                sharp = transforms.functional.vflip(
                    sharp
                )


            # ------------------------------------------------
            # Rotation
            # ------------------------------------------------

            angle = random.uniform(
                -10.0,
                10.0
            )

            blurred = transforms.functional.rotate(
                blurred,
                angle,
                interpolation=transforms.InterpolationMode.BICUBIC
            )

            sharp = transforms.functional.rotate(
                sharp,
                angle,
                interpolation=transforms.InterpolationMode.BICUBIC
            )


        # ====================================================
        # CONVERT TO TENSOR
        # ====================================================

        blurred = transforms.functional.to_tensor(
            blurred
        )

        sharp = transforms.functional.to_tensor(
            sharp
        )


        # ====================================================
        # IMAGENET NORMALIZATION
        # ====================================================

        blurred = (
            blurred - self.mean
        ) / self.std

        sharp = (
            sharp - self.mean
        ) / self.std


        # ====================================================
        # RETURN
        # ====================================================
        #
        # Input:
        #     Real blurred image
        #
        # Target:
        #     Corresponding real sharp image
        #
        # NO SYNTHETIC BLUR IS APPLIED.
        # ====================================================

        return blurred, sharp


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = PairedDeblurDataset(
    train_pairs,
    training=True
)

val_dataset = PairedDeblurDataset(
    val_pairs,
    training=False
)

test_dataset = PairedDeblurDataset(
    test_pairs,
    training=False
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("DATALOADER SUMMARY")
print("=" * 70)

print(
    f"Training pairs      : "
    f"{len(train_dataset)}"
)

print(
    f"Validation pairs    : "
    f"{len(val_dataset)}"
)

print(
    f"Test pairs          : "
    f"{len(test_dataset)}"
)

print(
    f"Training batches    : "
    f"{len(train_loader)}"
)

print(
    f"Validation batches  : "
    f"{len(val_loader)}"
)

print(
    f"Test batches        : "
    f"{len(test_loader)}"
)

print("=" * 70)


Searching GoPro directory:
/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/train
Detected direct blur/sharp structure.
Blur images  : 2103
Sharp images : 2103
Matched GoPro pairs: 2103

Searching GoPro directory:
/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test
Detected direct blur/sharp structure.
Blur images  : 1111
Sharp images : 1111
Matched GoPro pairs: 1111

RealBlur-J blur/sharp directory pairs found: 1
Matched RealBlur-J pairs: 980


FINAL DATASET SPLIT
GoPro official train pairs : 2103
GoPro train split           : 1892
GoPro validation split      : 210
GoPro official test pairs   : 1111
----------------------------------------------------------------------
RealBlur-J total pairs      : 980
RealBlur-J train split      : 686
RealBlur-J validation split : 147
RealBlur-J test split       : 147
----------------------------------------------------------------------
TOTAL TRAININ

# Phase 3: Model Architecture and Training Components

Define the residual deblurring network, perceptual feature extractor, loss function and optimization strategy used during training. Unlike the Super Resolution model, this architecture restores images at their original resolution and does not use adversarial training.

In [3]:
# ============================================================
# Residual Block (EDSR Style)
# ============================================================
class ResidualBlock(nn.Module):
    def __init__(self, channels, residual_scale=0.2):
        super().__init__()
        self.residual_scale = residual_scale
        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.PReLU(),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                stride=1,
                padding=1
            )
        )

    def forward(self, x):
        return x + self.block(x) * self.residual_scale

# ============================================================
# Deblur Generator
# ============================================================
class DeblurGenerator(nn.Module):
    def __init__(self, num_residual_blocks=16):
        super().__init__()
        # Initial Feature Extraction
        self.initial = nn.Sequential(
            nn.Conv2d(
                3,
                64,
                kernel_size=9,
                stride=1,
                padding=4
            ),
            nn.PReLU()
        )

        # Residual Trunk
        self.residuals = nn.Sequential(
            *[
                ResidualBlock(64)
                for _ in range(num_residual_blocks)
            ]
        )

        # Feature Fusion
        self.fusion = nn.Sequential(
            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(64)
        )

        # Reconstruction
        self.reconstruction = nn.Conv2d(
            64,
            3,
            kernel_size=9,
            stride=1,
            padding=4
        )

    def forward(self, x):
        initial = self.initial(x)
        x = self.residuals(initial)
        x = self.fusion(x)
        x = x + initial
        x = self.reconstruction(x)
        return x

# ============================================================
# Initialize Generator
# ============================================================
generator = DeblurGenerator(
    num_residual_blocks=NUM_RESIDUAL_BLOCKS
).to(device)

# ============================================================
# VGG19 Feature Extractor
# ============================================================
class VGGFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        weights = VGG19_Weights.IMAGENET1K_V1
        vgg = models.vgg19(weights=weights)
        self.features = nn.Sequential(
            *list(vgg.features.children())[:36]
        )

        for parameter in self.features.parameters():
            parameter.requires_grad = False
        self.features.eval()

    def forward(self, x):
        return self.features(x)
        
vgg = VGGFeatureExtractor().to(device)

# ============================================================
# SSIM Loss
# ============================================================
class SSIMLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.metric = StructuralSimilarityIndexMeasure(
            data_range=1.0
        ).to(device)

        self.mean = torch.tensor(
            IMAGENET_MEAN
        ).view(1,3,1,1).to(device)

        self.std = torch.tensor(
            IMAGENET_STD
        ).view(1,3,1,1).to(device)

    def denormalize(self, x):
        return torch.clamp(x * self.std + self.mean, 0, 1)

    def forward(self, prediction, target):
        prediction = self.denormalize(prediction)
        target = self.denormalize(target)
        return 1 - self.metric(prediction, target)

# ============================================================
# Deblur Loss
# ============================================================
class DeblurLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.perceptual = nn.L1Loss()
        self.ssim = SSIMLoss()

    def forward(self, prediction, target):
        l1_loss = self.l1(
            prediction,
            target
        )

        perceptual_loss = self.perceptual(
            vgg(prediction),
            vgg(target)
        )

        ssim_loss = self.ssim(
            prediction,
            target
        )

        total_loss = (
            0.80 * l1_loss +
            0.15 * perceptual_loss +
            0.05 * ssim_loss
        )

        return total_loss

# ============================================================
# Optimizer
# ============================================================
optimizer = optim.Adam(
    generator.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    weight_decay=WEIGHT_DECAY
)

# ============================================================
# Learning Rate Scheduler
# ============================================================
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10,
    min_lr=1e-6
)

# ============================================================
# Model Summary
# ============================================================
total_params = sum(
    p.numel()
    for p in generator.parameters()
)

trainable_params = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)

print("=" * 60)
print("Model Summary")
print("=" * 60)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Learning Rate        : {LEARNING_RATE}")
print(f"Residual Blocks      : {NUM_RESIDUAL_BLOCKS}")

# ============================================================
# Initialize Loss Function
# ============================================================
criterion = DeblurLoss().to(device)
print("Loss function initialized.")

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 191MB/s] 


Model Summary
Total Parameters     : 1,249,940
Trainable Parameters : 1,249,940
Learning Rate        : 0.0001
Residual Blocks      : 16
Loss function initialized.


# Phase 4: Model Training and Validation

Train the deblurring network using the generated training pairs, evaluate its performance on the validation set using Loss, PSNR and SSIM, and save the best-performing model together with resumable checkpoints.

**Resuming Training**

In [4]:
# ============================================================
# Resume Training
# ============================================================
checkpoint_path = "/kaggle/input/models/abdulwahib480/deblur/pytorch/default/16/checkpoint_latest.pth"

start_epoch = 0
best_validation_loss = float("inf")
train_losses = []
val_losses = []
psnr_scores = []
ssim_scores = []

if checkpoint_path is not None and os.path.exists(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device)
    
    generator.load_state_dict(checkpoint["generator_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    
    start_epoch = checkpoint["epoch"]
    #start_epoch += 1
    best_validation_loss = checkpoint["best_validation_loss"]
    train_losses = checkpoint["train_losses"]
    val_losses = checkpoint["val_losses"]
    psnr_scores = checkpoint["psnr_scores"]
    ssim_scores = checkpoint["ssim_scores"]

    print("=" * 60)
    print("Checkpoint Loaded Successfully")
    print("=" * 60)
    print(f"Resuming From Epoch : {start_epoch + 1}")
    print(f"Best Val Loss      : {best_validation_loss:.6f}")

else:
    print("=" * 60)
    print("No Checkpoint Found")
    print("Starting Training From Scratch")
    print("=" * 60)

Checkpoint Loaded Successfully
Resuming From Epoch : 181
Best Val Loss      : 0.094485


**Training Loop**

In [5]:
# ============================================================
# DENORMALIZE TENSOR
# ============================================================

def denormalize(tensor):
    """
    Convert ImageNet-normalized tensor back to [0, 1].

    Input:
        Tensor normalized using:
        (image - IMAGENET_MEAN) / IMAGENET_STD

    Output:
        Tensor in [0, 1]
    """

    mean = torch.tensor(
        IMAGENET_MEAN,
        device=tensor.device,
        dtype=tensor.dtype
    ).view(
        1, 3, 1, 1
    )

    std = torch.tensor(
        IMAGENET_STD,
        device=tensor.device,
        dtype=tensor.dtype
    ).view(
        1, 3, 1, 1
    )

    tensor = (
        tensor * std
        + mean
    )

    return tensor.clamp(
        0.0,
        1.0
    )

In [6]:
# ============================================================
# Model Saving Paths
# ============================================================
SAVE_DIR = "/kaggle/working/Deblur_Resume"
os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

best_model_path = os.path.join(
    SAVE_DIR,
    "best_generator.pth"
)

final_model_path = os.path.join(
    SAVE_DIR,
    "final_generator.pth"
)

checkpoint_path = os.path.join(
    SAVE_DIR,
    "checkpoint_latest.pth"
)


# ============================================================
# Training Loop
# ============================================================
for epoch in range(start_epoch, NUM_EPOCHS):
    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    generator.train()
    running_train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch [{epoch+1}/{NUM_EPOCHS}]"
    )

    for blurred, sharp in train_bar:
        blurred = blurred.to(device, non_blocking=True)
        sharp = sharp.to(device, non_blocking=True)
        optimizer.zero_grad()
        prediction = generator(blurred)
        loss = criterion(
            prediction,
            sharp
        )
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
        train_bar.set_postfix(
            Train_Loss=f"{loss.item():.4f}"
        )
        
    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------
    generator.eval()
    running_val_loss = 0.0
    running_psnr = 0.0
    running_ssim = 0.0
    
    with torch.no_grad():
        for blurred, sharp in val_loader:
            blurred = blurred.to(device, non_blocking=True)
            sharp = sharp.to(device, non_blocking=True)

            prediction = generator(blurred)

            loss = criterion(
                prediction,
                sharp
            )

            running_val_loss += loss.item()
            prediction_metric = denormalize(prediction)
            sharp_metric = denormalize(sharp)

            batch_psnr = psnr_metric(
                prediction_metric,
                sharp_metric
            )

            batch_ssim = ssim_metric(
                prediction_metric,
                sharp_metric
            )
            running_psnr += batch_psnr.item()
            running_ssim += batch_ssim.item()

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_psnr = running_psnr / len(val_loader)
    epoch_ssim = running_ssim / len(val_loader)
    val_losses.append(epoch_val_loss)
    psnr_scores.append(epoch_psnr)
    ssim_scores.append(epoch_ssim)

    # --------------------------------------------------------
    # Scheduler Update
    # --------------------------------------------------------
    scheduler.step(epoch_val_loss)

    # --------------------------------------------------------
    # Save Best Generator
    # --------------------------------------------------------
    if epoch_val_loss < best_validation_loss:
        best_validation_loss = epoch_val_loss
        torch.save(generator.state_dict(), best_model_path)
        best_epoch = epoch + 1
        print(f"Best model saved at epoch {best_epoch}")

    # --------------------------------------------------------
    # Epoch Logging
    # --------------------------------------------------------
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"Learning Rate : {current_lr:.8f}")
    print(f"Train Loss    : {epoch_train_loss:.6f}")
    print(f"Valid Loss    : {epoch_val_loss:.6f}")
    print(f"PSNR          : {epoch_psnr:.4f}")
    print(f"SSIM          : {epoch_ssim:.4f}")
    print(f"Best Val Loss : {best_validation_loss:.6f}\n")

# ============================================================
# Save Final Generator After Training
# ============================================================
torch.save(
    generator.state_dict(),
    final_model_path
)

print("\nTraining Completed.")
print(f"Best Generator : {best_model_path}")
print(f"Final Generator: {final_model_path}")

Epoch [181/200]:   0%|          | 0/322 [00:00<?, ?it/s]

Epoch [181/200]
Learning Rate : 0.00010000
Train Loss    : 0.098312
Valid Loss    : 0.095567
PSNR          : 27.8776
SSIM          : 0.8802
Best Val Loss : 0.094485



Epoch [182/200]:   0%|          | 0/322 [00:00<?, ?it/s]

Epoch [182/200]
Learning Rate : 0.00010000
Train Loss    : 0.097816
Valid Loss    : 0.096587
PSNR          : 27.8839
SSIM          : 0.8808
Best Val Loss : 0.094485



Epoch [183/200]:   0%|          | 0/322 [00:00<?, ?it/s]

Epoch [183/200]
Learning Rate : 0.00010000
Train Loss    : 0.097806
Valid Loss    : 0.094572
PSNR          : 27.9833
SSIM          : 0.8820
Best Val Loss : 0.094485



... (30 outputs left out)

Epoch [199/200]:   0%|          | 0/322 [00:00<?, ?it/s]

Epoch [199/200]
Learning Rate : 0.00010000
Train Loss    : 0.096606
Valid Loss    : 0.094341
PSNR          : 28.0964
SSIM          : 0.8812
Best Val Loss : 0.093733



Epoch [200/200]:   0%|          | 0/322 [00:00<?, ?it/s]

Best model saved at epoch 200
Epoch [200/200]
Learning Rate : 0.00010000
Train Loss    : 0.096163
Valid Loss    : 0.093400
PSNR          : 28.1626
SSIM          : 0.8847
Best Val Loss : 0.093400


Training Completed.
Best Generator : /kaggle/working/Deblur_Resume/best_generator.pth
Final Generator: /kaggle/working/Deblur_Resume/final_generator.pth


**Saving Training**

In [7]:
import os
# ============================================================
# Manual Checkpoint Saving
# ============================================================
checkpoint_path = os.path.join(
    SAVE_DIR,
    "checkpoint_latest.pth"
)

checkpoint = {
    # Current training position
    "epoch": epoch,
    # Model
    "generator_state_dict": generator.state_dict(),
    # Optimization
    "optimizer_state_dict": optimizer.state_dict(),
    # Scheduler
    "scheduler_state_dict": scheduler.state_dict(),
    # Best validation tracking
    "best_validation_loss": best_validation_loss,
    # Training history
    "train_losses": train_losses,
    "val_losses": val_losses,
    "psnr_scores": psnr_scores,
    "ssim_scores": ssim_scores
}

torch.save(
    checkpoint,
    checkpoint_path
)

print("=" * 60)
print("Checkpoint Saved Successfully")
print("=" * 60)
print(f"Saved Epoch       : {epoch + 1}")
print(f"Best Val Loss     : {best_validation_loss:.6f}")
print(f"Checkpoint Path   : {checkpoint_path}")

Checkpoint Saved Successfully
Saved Epoch       : 200
Best Val Loss     : 0.093400
Checkpoint Path   : /kaggle/working/Deblur_Resume/checkpoint_latest.pth
